In [1]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
# --- 경로 설정 ---
DATASET_DIR = r'/content/drive/MyDrive/말동이_project/voice/final_training_data_n202'
MODEL_SAVE_DIR = r'/content/drive/MyDrive/말동이_project/voice/models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
model_checkpoint_path = os.path.join(MODEL_SAVE_DIR, 'cnn_bilstm_model_n202.pt') # PyTorch 모델 확장자

# --- 설정 변수 ---
EPOCHS = 200
BATCH_SIZE = 16
L2_REG = 0.0008
DROP_RATE = 0.6
LEARNING_RATE = 0.0001
PATIENCE = 20  # EarlyStopping patience
LR_PATIENCE = 10 # ReduceLROnPlateau patience

# GPU 설정
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {DEVICE}")

# =========================================================================
# 2. 데이터 로드 및 DataLoader 생성
# =========================================================================
def load_data_and_create_dataloader(data_dir, batch_size):
    print(f"\n데이터 로드 시작: {data_dir}")
    # NumPy 배열 로드 및 float32로 변환
    X_train = np.load(os.path.join(data_dir, 'X_train_padded.npy')).astype(np.float32)
    X_val = np.load(os.path.join(data_dir, 'X_val_padded.npy')).astype(np.float32)
    X_test = np.load(os.path.join(data_dir, 'X_test_padded.npy')).astype(np.float32)
    y_train = np.load(os.path.join(data_dir, 'y_train_ohe.npy')).astype(np.float32)
    y_val = np.load(os.path.join(data_dir, 'y_val_ohe.npy')).astype(np.float32)
    y_test = np.load(os.path.join(data_dir, 'y_test_ohe.npy')).astype(np.float32)

    # PyTorch Conv1D에 맞게 축 전치: (batch, L, C) -> (batch, C, L)
    X_train_tensor = torch.tensor(X_train).permute(0, 2, 1)
    X_val_tensor = torch.tensor(X_val).permute(0, 2, 1)
    X_test_tensor = torch.tensor(X_test).permute(0, 2, 1)

    y_train_tensor = torch.tensor(y_train)
    y_val_tensor = torch.tensor(y_val)
    y_test_tensor = torch.tensor(y_test)

    # Dataset 및 DataLoader 생성
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    INPUT_CHANNELS = X_train_tensor.shape[1] # C = features (202)
    INPUT_LENGTH = X_train_tensor.shape[2] # L = sequence_length
    NUM_CLASSES = y_train_tensor.shape[1]

    print(f"X_train shape (PyTorch format C,L): {X_train_tensor.shape}")
    print(f"입력 채널 (INPUT_CHANNELS): {INPUT_CHANNELS}")
    print(f"입력 길이 (INPUT_LENGTH): {INPUT_LENGTH}")
    print(f"클래스 개수 (NUM_CLASSES): {NUM_CLASSES}")
    return train_loader, val_loader, test_loader, INPUT_CHANNELS, INPUT_LENGTH, NUM_CLASSES

# 데이터 로드 실행
train_loader, val_loader, test_loader, INPUT_CHANNELS, INPUT_LENGTH, NUM_CLASSES = load_data_and_create_dataloader(DATASET_DIR, BATCH_SIZE)

# =========================================================================
# 3. 🧠 PyTorch 모델 정의 (2-Block CNN-BiLSTM)
# =========================================================================
class CNNBiLSTM_Prosody(nn.Module):
    def __init__(self, input_channels, num_classes, drop_rate):
        super(CNNBiLSTM_Prosody, self).__init__()

        # 1. CNN Block 1 (변경 없음)
        self.conv1 = nn.Conv1d(input_channels, 256, kernel_size=5, padding='same', bias=False)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2, padding=1)
        self.dropout1 = nn.Dropout(drop_rate)

        # 2. CNN Block 2 (변경 없음)
        self.conv2 = nn.Conv1d(256, 256, kernel_size=3, padding='same', bias=False)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2, padding=1)
        self.dropout2 = nn.Dropout(drop_rate)


        # 3. Stacked Bi-LSTM (히든 사이즈 증가)
        self.lstm1 = nn.LSTM(256, 128, batch_first=True, bidirectional=True)
        self.dropout3 = nn.Dropout(drop_rate)

        # lstm2 입력 크기
        self.lstm2 = nn.LSTM(128 * 2, 64, batch_first=True, bidirectional=True)
        self.dropout4 = nn.Dropout(drop_rate)

        # 4. Dense Classification (입력 및 출력 크기 변경)
        # Dense1 입력 크기
        # Dense1 출력 크기
        self.dense1 = nn.Linear(64 * 2, 64)
        self.dropout5 = nn.Dropout(drop_rate)

        # 최종 출력층 입력 크기
        self.output_layer = nn.Linear(64, num_classes)

    def forward(self, x):


        # CNN Blocks (변경 없음)
        x = self.conv1(x); x = self.bn1(x); x = self.relu(x); x = self.pool1(x); x = self.dropout1(x)
        x = self.conv2(x); x = self.bn2(x); x = self.relu(x); x = self.pool2(x); x = self.dropout2(x)

        # LSTM 입력 형태 변환
        x = x.permute(0, 2, 1)

        # Stacked Bi-LSTM (변경 없음)
        x, _ = self.lstm1(x)
        x = self.dropout3(x)

        _, (h_n, c_n) = self.lstm2(x)

        # 최종 Bi-LSTM 출력 (순방향과 역방향의 마지막 hidden state를 연결)
        # h_n[-2]와 h_n[-1]을 사용하여 연결. 출력 shape: [batch, 384 * 2 = 768]
        forward_hidden = h_n[-2, :, :]
        backward_hidden = h_n[-1, :, :]
        x = torch.cat((forward_hidden, backward_hidden), dim=1)

        # Dense Classification (변경 없음)
        x = self.relu(self.dense1(x))
        x = self.dropout5(x)

        x = self.output_layer(x)

        return x

# 모델 생성 및 장치로 이동
model = CNNBiLSTM_Prosody(INPUT_CHANNELS, NUM_CLASSES, DROP_RATE).to(DEVICE)

# 옵티마이저 설정 (L2 정규화는 weight_decay 인자로 대체)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=L2_REG)
criterion = nn.CrossEntropyLoss()

# 모델 구조 출력
print("\n--- PyTorch CNN-BiLSTM 모델 구조 ---")
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 학습 가능한 파라미터 수: {total_params}")

# =========================================================================
# 4. 🚀 학습 함수 및 루프 정의
# =========================================================================

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, torch.argmax(labels, dim=1))

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == torch.argmax(labels, dim=1)).sum().item()

    return running_loss / total_samples, correct_predictions / total_samples

def evaluate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, torch.argmax(labels, dim=1))

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == torch.argmax(labels, dim=1)).sum().item()

    return running_loss / total_samples, correct_predictions / total_samples

# 콜백 설정 (PyTorch 스케줄러) - verbose 인자 제거
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=LR_PATIENCE, min_lr=1e-6)

# 학습 루프
print("\n--- 2-Block CNN-BiLSTM 학습 시작 (PyTorch) ---")
start_time = time.time()

best_val_accuracy = 0.0
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate_epoch(model, val_loader, criterion, DEVICE)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} | "
          f"LR: {current_lr:.6f}")

    # ModelCheckpoint (val_accuracy 기준)
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), model_checkpoint_path)
        print(f"Epoch {epoch}: val_accuracy 개선됨. 모델 저장 중... ({best_val_accuracy:.4f})")
    else:
        patience_counter += 1

    # ReduceLROnPlateau (val_loss 기준)
    scheduler.step(val_loss)

    # EarlyStopping (val_accuracy 기준)
    if patience_counter >= PATIENCE:
        print(f"\n조기 종료: {PATIENCE} 에포크 동안 val_accuracy 개선이 없어 학습을 중단합니다.")
        # Restore best weights
        model.load_state_dict(torch.load(model_checkpoint_path))
        print(f"최적 모델 가중치로 복원되었습니다.")
        break

end_time = time.time()
print(f"총 학습 시간: {(end_time - start_time)/60:.2f}분")

# --- 5. 최종 평가 ---
print("\n--- 최종 테스트 데이터셋 평가 ---")
test_loss, test_accuracy = evaluate_epoch(model, test_loader, criterion, DEVICE)

print(f"테스트 손실 (Test Loss): {test_loss:.4f}")
print(f"테스트 정확도 (Test Accuracy): {test_accuracy:.4f}")
print(f"\n🎉 최적 모델 가중치가 다음 경로에 저장되었습니다: {model_checkpoint_path}")

사용 장치: cuda

데이터 로드 시작: /content/drive/MyDrive/말동이_project/voice/final_training_data_n202
X_train shape (PyTorch format C,L): torch.Size([21000, 202, 286])
입력 채널 (INPUT_CHANNELS): 202
입력 길이 (INPUT_LENGTH): 286
클래스 개수 (NUM_CLASSES): 7

--- PyTorch CNN-BiLSTM 모델 구조 ---
CNNBiLSTM_Prosody(
  (conv1): Conv1d(202, 256, kernel_size=(5,), stride=(1,), padding=same, bias=False)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.6, inplace=False)
  (conv2): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=same, bias=False)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
  (dropout2): Dropout(p=0.6, inplace=False)
  (lstm1): LSTM(256, 128, batch_first=True, bidirectional=True)
  (dropout3): Dropout(